# Transfer pixel from a photo to recreate another photo

## Idea:
- just one fct to do it to get the new position in same frame,
then we can decide either to transfer pixel in same frame or to another one:
e.g. transfer pixel from frame on the left to frame on the right, or from upper frame to downside frame
=> we can place 4 famous paintings in a big canvas, then transfer pixels from each painting to their neighbor clockwise.

## Import modules

In [1]:
# import internal modules
from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
from dataclasses import dataclass

# import 3rd-party modules
import cv2
import numpy as np
from numba import njit
import random

# import local modules
from utils.renderer.giffer import create_gif
from utils.project_manager import Project
from utils.renderer.resizer import resize_with_pad, resize_with_crop

## Set up project

In [2]:
# check current working directory in notebook session
!pwd

/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core


In [3]:
# create project
project = Project(project_dir="assets/images/pixel_transfers", in_img_dir="db", out_img_dir_list=["quadruple_transfer"])

## Define functions and classes

In [4]:
# decorate function with numba fct to speed up execution
@njit()
def get_pixel_coords_map(src_img, dest_img):
    """
    Function to find best matching pixel for region of interest (a pixel) of another image
    Important: The images should be in LAB color space for better results.

    Arguments
    * src_img: source image
    * dest_img: destination image, i.e. image to recreate with pixel from source image
    """
    # create empty lists (or arrays) to store coords mapping between source and dest images
    src_yxs = []
    dest_yxs = []

    # get list of all pixel coords in src and dest images
    dest_pixel_coords = [(dest_y, dest_x) for dest_y in range(dest_img.shape[0]) for dest_x in range(dest_img.shape[1])]
    src_pixel_coords = [(src_y, src_x) for src_y in range(src_img.shape[0]) for src_x in range(src_img.shape[1])]

    # iterate over each pixel position
    for dest_y, dest_x in dest_pixel_coords:
        # get region of interest (pixel) in destination img
        dest_pixel = dest_img[dest_y, dest_x]


        # inititiate trackers for best match to this roi
        best_match_dist = np.inf
        best_match_index = 0

        # iterate over each pixel in source image
        for src_pixel_idx, (src_y, src_x) in enumerate(src_pixel_coords):
            # get source pixel
            src_pixel = src_img[src_y, src_x]

            # compute distance between the pixels
            dist = np.abs(dest_pixel - src_pixel)
            
            # if distance is smaller than the current best dist
            if dist < best_match_dist:
                # update current best dist
                best_match_dist = dist
                # store best match index 
                best_match_index = src_pixel_idx

        # take out best src pixel coords from list of src pixel coords
        (best_match_y, best_match_x) = src_pixel_coords.pop(best_match_index)

        # append best src pixel coords and their corresponding dest pixel coords to lists
        src_yxs.append((best_match_y, best_match_x))
        dest_yxs.append((dest_y, dest_x))

    return src_yxs, dest_yxs

## Get images paths

In [5]:
# # set img_path_list
# project.img_path_list = [
#     "/assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg",
#     "/assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg",
#     "core/assets/images/pixel_transfers/db/Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpeg",
#     "core/assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg"
# ]

In [6]:
# get img_path_list
project.in_img_path_list

['assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg',
 'assets/images/pixel_transfers/db/Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpeg',
 'assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg',
 'assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg']

In [7]:
# read first image to get a reference shape
ref_img_height, ref_img_width, ref_img_channel = cv2.imread(project.in_img_path_list[0]).shape

In [8]:
nb_imgs = len(project.in_img_path_list)

# get src-dest pair of img paths for mapping
src_dest_pairs = []
for img_idx in range(nb_imgs):
    src_dest_pairs.append((project.in_img_path_list[img_idx], project.in_img_path_list[(img_idx + 1)%nb_imgs]))
src_dest_pairs

[('assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg',
  'assets/images/pixel_transfers/db/Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpeg'),
 ('assets/images/pixel_transfers/db/Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpeg',
  'assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg'),
 ('assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg',
  'assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg'),
 ('assets/images/pixel_transfers/db/picasso-the-weeping-woman.jpeg',
  'assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg')]

## Get yx coords mapping from one photo to another

In [9]:
for src_img_path, dest_img_path in src_dest_pairs:
    src_img = resize_with_crop(img_path=src_img_path, ref_img_shape=(ref_img_height, ref_img_width, ref_img_channel))
    dest_img = resize_with_crop(img_path=dest_img_path, ref_img_shape=(ref_img_height, ref_img_width, ref_img_channel))

    # show alongside reference image and dest image with padding and resized
    cv2.imshow("result",cv2.hconcat([src_img, dest_img]))

    # wait for any press on keyboard
    cv2.waitKey(0)

    # destroy all windows
    cv2.destroyAllWindows()
    cv2.waitKey(1) # workaround on mac to effectively close the windows

    # src_yxs, dest_yxs = get_pixel_coords_map(src_img, dest_img)

img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641
img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641
img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641
img: (1000, 750), ref_img: (374, 266)
img_ratio: 0.75, ref_img_ratio: 0.7112299465240641
img: (1000, 750), ref_img: (374, 266)
img_ratio: 0.75, ref_img_ratio: 0.7112299465240641
img: (732, 601), ref_img: (374, 266)
img_ratio: 0.8210382513661202, ref_img_ratio: 0.7112299465240641
img: (732, 601), ref_img: (374, 266)
img_ratio: 0.8210382513661202, ref_img_ratio: 0.7112299465240641
img: (374, 266), ref_img: (374, 266)
img_ratio: 0.7112299465240641, ref_img_ratio: 0.7112299465240641


## Create images to visualize pixel transfer

## Create gif or video from output images

In [8]:
Path("ty").parent

PosixPath('.')